%md

para usar o databricks

https://dbc-0b69ef93-5247.cloud.databricks.com/editor/notebooks/2520840228223736?o=7474657872577658#command/8015241331793059

aulas
https://customer-academy.databricks.com/learn/courses/3901/introduction-to-apache-spark?hash=6788372f2d7294e77f5eb6191645568ae747f68b&generated_by=409740

# 1.3 Demo - Exploring Spark Architecture in Databricks

In this demonstration, we explore how Spark's architecture components manifest in a Databricks environment and how to monitor them using the various UIs available to us. Rather than just reading about the Driver, Executors, and the DAG, you will **run a real job, hunt for these components in the Spark UI, and record what you find**.

## Learning Objectives

**By the end of this demo, you will be able to:**

## REQUIRED - SELECT CLASSIC COMPUTE

> **Select Compute**
> 
> Before starting this notebook, select your **Classic Compute** cluster from the compute drop-down at the top-right of the notebook. Be aware that **Serverless** is selected by default.
> 
> * If your cluster is not shown in the drop-down, select **More**, then choose your cluster in the **Attach to an existing compute resource** dialog.
> * If your cluster has terminated, open **Compute** in the left navigation, start your cluster, then return here and attach it.
> 
> **NOTE:** This course was developed and tested on **Databricks Runtime 17.3 LTS (Apache Spark 4.0)**. Several demonstrations explore the Spark UI and cluster internals, which are not exposed on Serverless compute.

---



## A. Introduction to Notebooks

We will run all of the demos and exercises for this course in notebooks. Notebooks are interactive documents that combine live code, visualizations, narrative text, and outputs in a single document.

Key features of Databricks notebooks:

* **Code execution:** run code in multiple languages (Python, SQL, R, Scala)
* **Rich text formatting:** support for Markdown to create well-documented analyses
* **Cell-based structure:** code and content are organized into executable cells
* **Interactive visualizations:** direct plotting and charting of results
* **Collaboration:** share and work together on notebooks in real time


The default language for a notebook is shown in the toolbar above. The notebooks in this course default to SQL, and we use magic commands to run other languages within the same notebook:

* Use `%python` to execute Python code
* Use `%sql` to execute SQL queries
* Use `%r` to execute R code
* Use `%scala` to execute Scala code

The `%run` magic command is particularly useful - it executes another notebook within your current notebook, enabling modular code organization and reuse. The demos in this course use `%run` to invoke their classroom setup notebooks.

---

## B. The SparkSession and SparkContext

The `SparkSession` is your entry point to everything in Spark, and Databricks instantiates it for you automatically as the variable `spark`. The older `SparkContext` sits beneath it. Whether you can reach the `SparkContext` directly depends on the **access mode** of your compute - we will detect yours in a moment and see for ourselves.

In [0]:
%python
# Display the SparkSession
spark

In [0]:
%python
# Confirm the Spark Version
print(f"This cluster is running Apache Spark {spark.version}")

This cluster is running Apache Spark 4.2.0


> **Spark 4.0**
> 
> The version printed above should start with `4.` - Databricks Runtime 17.x is built on Apache Spark 4.0. Everything in this course uses Spark 4 behavior, including ANSI SQL mode by default.

In [0]:
%python
# Get the Application ID
# spark.conf is Spark Connect safe; spark.sparkContext.applicationId is not
#application_id = spark.conf.get("spark.app.id")
try:
    application_id = spark.conf.get("spark.app.id")
    print(f"Application ID: {application_id}")
except Exception:
    print("Application ID: not available on this compute.")
    print("Serverless exposes only a subset of runtime config keys via spark.conf;")
    print("attach a Classic Compute cluster to retrieve the application ID.")

Application ID: not available on this compute.
Serverless exposes only a subset of runtime config keys via spark.conf;
attach a Classic Compute cluster to retrieve the application ID.


In [0]:
# Display the Spark Configuration
# SQL "SET" lists the runtime configuration exposed by this compute.
# It works on both Classic and Serverless access modes (spark.conf.get()
# blocks most keys on Serverless, as we saw in the previous cell).
config_df = spark.sql("SET")
display(config_df)

key,value
spark.databricks.execution.timeout,9000
spark.sql.ansi.enabled,true
spark.sql.shuffle.partitions,auto


In [0]:
%python
# The SparkSession is a Managed Singleton
from pyspark.sql import SparkSession

# In Databricks the session already exists; getOrCreate() returns that same
# session rather than constructing a new one
print(f"getOrCreate() returns the existing session: {SparkSession.builder.getOrCreate() is spark}")
print(f"An active session is set: {SparkSession.getActiveSession() is not None}")

getOrCreate() returns the existing session: True
An active session is set: True


In [0]:
%python
# Detect Your Compute's Access Mode
# Whether `spark` uses the Spark Connect API tells you how this session talks
# to Spark. It does not by itself determine whether the JVM-side SparkContext
# (sc) is exposed. On recent DBR, dedicated-access Classic clusters also serve
# `spark` via Spark Connect while still injecting a driver-side `sc`.
uses_spark_connect = "connect" in type(spark).__module__

if uses_spark_connect:
    print("The `spark` session uses the Spark Connect API.")
    print("This is the case on standard-access/shared, Serverless, and recent dedicated-access clusters.")
else:
    print("The `spark` session is a classic (JVM-attached) SparkSession.")

print("The next cell probes `sc` directly to determine whether the SparkContext is exposed.")

The `spark` session uses the Spark Connect API.
This is the case on standard-access/shared, Serverless, and recent dedicated-access clusters.
The next cell probes `sc` directly to determine whether the SparkContext is exposed.


In [0]:
%python
# Try to Access the SparkContext
# Probing `sc` directly tells you whether this session runs with a driver-side
# SparkContext. Note this is independent of whether `spark` uses the Spark
# Connect API - the previous cell showed dedicated clusters can use both.
try:
    print(sc)
    print("\nSparkContext (sc) is available.")
    print("The notebook is attached to the driver JVM - this is dedicated-access (single-user) compute.")
except NameError as e:
    print("SparkContext (sc) is not defined in this session.")
    print("This is standard-access/shared or Serverless compute, where the notebook")
    print("only holds a Spark Connect session and the JVM-side SparkContext is not exposed.")
    print(f"\n{type(e).__name__}: {e}")


SparkContext (sc) is available.
The notebook is attached to the driver JVM - this is dedicated-access (single-user) compute.


> **The SparkContext is Legacy**
> 
> The `SparkContext` (`sc`) predates the `SparkSession`. On a **dedicated-access Classic cluster** - the compute this demo asks you to use - it is still reachable as `sc` or `spark.sparkContext`, which is why the cell above printed the object. On compute that uses **Spark Connect** (standard-access/shared clusters and Serverless), the `SparkContext` and the RDD API are not exposed, and the same cell would print a message instead. Either way, modern Spark code works through the `SparkSession` and DataFrames - you rarely touch the `SparkContext` directly.

## C. Understanding Your Databricks Cluster

In the runtime architecture lecture, the **Cluster Manager** allocated resources to the Driver. On Databricks you never run that Cluster Manager yourself - **the platform plays that role for you**. When you create a cluster, Databricks provisions a Driver node and Worker nodes, wires them together, and hands you a ready-to-use `SparkSession`. Let's see where that shows up in the UI.

1. **Right-click on the Compute** menu item in the left navigation pane and select **Open link in new tab**
2. Locate your cluster in the **All-purpose compute** tab. Here you can see high-level information about the cluster including memory and cores
3. Click on the link to the cluster under the Name column. Here you can see more detailed information and access logs and metrics

> **Record This**
> 
> On the cluster page, find and note down: **(a)** how many Worker nodes your cluster has, and **(b)** the total number of cores. In a single-node training cluster the Driver and Executor may share one machine - this is expected. You'll compare these numbers against the Spark UI later.

---

## D. Creating and Monitoring a Spark Job

Let's create a simple job that will help us visualize the execution flow. Run the cell below to create a Spark job.

> **Focus on the Execution, Not the Code**
> 
> Don't focus too much on the code here. We want to explore the logs and metrics this job produces. The DataFrame operations themselves are covered in the next section.

In [0]:
%python
# Run a Job with a Shuffle and a Cache
# Import the functions module as F - the recommended pattern. Importing names like
# sum directly would shadow Python built-ins, so we always qualify them as F.sum, F.col, etc.
import pyspark.sql.functions as F

# Simulate 20 million e-commerce order line items.
# This is a realistic-looking dataset, not abstract numbers: every row is one
# order placed by a customer, for some amount, in some product category.
NUM_ROWS = 20_000_000

df = (
    spark.range(NUM_ROWS)
    # 500,000 distinct customers -> a HIGH-cardinality key (this is what makes the shuffle big)
    .withColumn("customer_id", (F.rand(seed=1) * 500_000).cast("int")) # rand(seed=1) cria valores aleatórios
    # monetary amount per order
    .withColumn("amount", F.round(F.rand(seed=2) * 500, 2))
    # a text payload, so each row carries real bytes (fills the cache in the Storage tab)
    .withColumn("category", F.concat(F.lit("product_category_"), (F.rand(seed=3) * 50).cast("int")))
)

# Cache the raw dataset so the Storage tab shows a large (hundreds of MB) cached table
try:
    df.cache()
except Exception:
    print("df.cache() is not supported on Serverless compute - caching is managed automatically.")
    print("Attach a Classic Compute cluster to cache the dataset and inspect the Storage tab.")

df.cache() is not supported on Serverless compute - caching is managed automatically.
Attach a Classic Compute cluster to cache the dataset and inspect the Storage tab.


 on Serverless, .cache() is not available because caching is managed automatically. The DataFrame definition itself is fine; only the cache call fails. I'll guard that single call so the cell runs on Serverless while preserving the caching behavior on Classic.

In [0]:
df.show()

+---+-----------+------+-------------------+
| id|customer_id|amount|           category|
+---+-----------+------+-------------------+
|  0|       5460| 178.3| product_category_9|
|  1|     442976|203.92| product_category_1|
|  2|      79222| 92.18|product_category_30|
|  3|     360910|134.78|product_category_31|
|  4|     173769| 80.91|product_category_32|
|  5|      73770|176.86| product_category_4|
|  6|     102185|317.65|product_category_49|
|  7|     466450|391.72|product_category_16|
|  8|     459132|223.66|product_category_10|
|  9|     225172| 97.19|product_category_48|
| 10|     379397|414.14|product_category_27|
| 11|     149035| 12.67|product_category_32|
| 12|     357600| 157.5|product_category_22|
| 13|     465616|411.48|product_category_35|
| 14|      91376|253.01|product_category_13|
| 15|     180884|375.33|product_category_20|
| 16|     315761| 41.78|product_category_10|
| 17|     126238|202.08| product_category_8|
| 18|     233952| 477.7| product_category_1|
| 19|     

Cria um DataFrame Spark com 20 milhões de linhas através de spark.range().  

Adiciona colunas simulando transações de e-commerce: customer_id (cerca de 500.000 clientes distintos), amount (valores monetários aleatórios) e category (uma string formatada para ocupar espaço em memória).   

In [0]:
df.count()

20000000

In [0]:

# The shuffle: total revenue per customer.
# Because there are ~500,000 distinct customers, Spark CANNOT collapse the data
# locally on each partition - it must send every customer's rows across the
# network so they land on the same executor. That network exchange is the
# shuffle, and here it moves well over 100 MB (check "Shuffle Write" in the UI).
result = (
    df.groupby("customer_id")
    .agg(
        F.sum("amount").alias("total_spend"),
        F.count("*").alias("num_orders"),
    )
)

# Trigger the job with an action; we count the resulting groups rather than

In [0]:
result.show()

+-----------+------------------+----------+
|customer_id|       total_spend|num_orders|
+-----------+------------------+----------+
|      33338|          11388.82|        42|
|     307781|11993.289999999999|        53|
|      40899| 9853.590000000002|        35|
|     408327|11706.400000000001|        46|
|     243475|          10785.66|        43|
|      46648|13923.650000000001|        52|
|      29039|          12308.66|        50|
|     290831| 9628.300000000001|        39|
|      73145|          12294.06|        45|
|     429624| 9293.900000000001|        40|
|     335873|           9751.32|        40|
|     295815|          18756.64|        67|
|     247897|            9740.4|        41|
|     254816|          13049.55|        57|
|     141078|13659.419999999998|        52|
|     192138|          10960.15|        42|
|     377511| 9565.289999999999|        39|
|     459374|           10276.3|        46|
|     454895| 9539.160000000002|        38|
|       4101| 6497.759999999999|

In [0]:
result.count()

500000

In [0]:
# Trigger the job with an action. We count the resulting groups rather than
# collect() them, so we never pull 500,000 rows back to the Driver.
num_customers = result.count()
print(f"Computed revenue for {num_customers:,} customers from {NUM_ROWS:,} orders.")
print("Open the Spark UI and look at the Shuffle Write / Shuffle Read for this job - it should be 100+ MB.")

Computed revenue for 500,000 customers from 20,000,000 orders.
Open the Spark UI and look at the Shuffle Write / Shuffle Read for this job - it should be 100+ MB.
